# Figure 11 (1/3): reference curve κ0(φ)

**Permeability of the undeformed hexagonal-pore unit cells**

This notebook computes the homogenized permeability κ0 of the undeformed unit cells with hexagonal pores, for $r_0 = 0.02, 0.03, \dots, 0.50$ (49 unit cells), by one-step linear homogenization (matrix permeability $k_l = \mathbf{1}$, probe pressure gradient 0.01), on two meshes:

- default mesh (mesh size $l = 0.03$, as in Fig. 3 and Fig. 10; UMFPACK): probes $g_x$ and $g_y$, 98 runs;
- refined mesh ($l = 0.015$, refined to $r_0/10$ near the pores; MUMPS): probe $g_x$, plus the probe $g_y$ for $r_0 = 0.07$ and $0.30$ (isotropy check), 51 runs.

The curves κ0(φ) (PCHIP interpolation in the mesh porosity) are used by `Fig11.ipynb`. The runs are executed one after the other; the full notebook takes about one hour.

---

**Authors**

Haotian XIAO
Martin GENET

École Polytechnique, Palaiseau, France

## Imports

In [ ]:
from pathlib import Path

import Runs_Microporoflow as rm
import Post_Microporoflow as pm

# Quieter output (messages only, results unchanged; see Quiet_Microporoflow.py):
# no gmsh information/progress lines, no ufl FutureWarning
import Quiet_Microporoflow as qm

qm.quiet_fe_output()

## Output directory

In [ ]:
# All results of this notebook are written to the subdirectory Fig11-kappa0/
# of the notebook directory (demos/).
RESULTS_DIR = Path("Fig11-kappa0").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR =", qm.display_path(RESULTS_DIR))

## Parameters

In [ ]:
do_run_default = 1  # finite-element runs on the default mesh (98 runs)
do_run_refined = 1  # finite-element runs on the refined mesh (51 runs)
do_check       = 1  # post-processing only: kappa0 on both meshes at the Fig. 10 r0, isotropy

r0_list = [k / 100 for k in range(2, 51)]  # 0.02, 0.03, ..., 0.50
assert all(r0 == float(f"{r0:.2f}") for r0 in r0_list)

# Default mesh (l = 0.03, UMFPACK): every r0 with the probes gx and gy
default_tag = "linear_hex_l0.03"
cases_default = [
    dict(r0=r0, probe=probe, l=0.03, local=False, solver="umfpack")
    for r0 in r0_list
    for probe in ("gx", "gy")
]

# Refined mesh (l = 0.015, refined to r0/10 near the pores, MUMPS): every r0 with the probe gx,
# then the probe gy for r0 = 0.07 and 0.3
refined_tag = "linear_hex_l0.015_local"
cases_refined = [
    dict(r0=r0, probe="gx", l=0.015, local=True, solver="mumps")
    for r0 in r0_list
] + [
    dict(r0=r0, probe="gy", l=0.015, local=True, solver="mumps")
    for r0 in (0.07, 0.3)
]

print(len(cases_default), "default-mesh runs,", len(cases_refined), "refined-mesh runs")
print("material parameters:", rm.MATERIAL_PARAMETERS)

## Computing κ0 on the default mesh

In [ ]:
if do_run_default:
    rm.run_case_list(rm.run_linear_hex_case, cases_default, RESULTS_DIR / default_tag)

## Computing κ0 on the refined mesh

In [ ]:
if do_run_refined:
    rm.run_case_list(rm.run_linear_hex_case, cases_refined, RESULTS_DIR / refined_tag)

## Check

In [ ]:
if do_check:
    pm.require_files({
        "Fig11-kappa0.ipynb (this notebook, do_run_default = do_run_refined = 1)":
            pm.kappa0_input_files(RESULTS_DIR / default_tag, False)
            + pm.kappa0_input_files(RESULTS_DIR / refined_tag, True, gy_r0=(0.07, 0.30)),
    })
    pm.kappa0_check(RESULTS_DIR / default_tag, RESULTS_DIR / refined_tag)